# Football Player Tracker — Optuna HPO (Colab)

Corre **20 trials** de búsqueda bayesiana (TPE) sobre learning rate, momentum, weight decay y augmentaciones.  
Cada trial entrena YOLOv11n durante **15 épocas** en T4, loguea métricas a MLflow como **nested runs** bajo un parent run de HPO, y persiste el estudio en SQLite para que sea resumible.

**Search space** (definido en `configs/tuning/optuna.yaml`):
| Param | Tipo | Rango |
|---|---|---|
| `lr0` | log-uniform | 1e-5 … 1e-2 |
| `momentum` | uniform | 0.85 … 0.98 |
| `weight_decay` | log-uniform | 1e-6 … 1e-3 |
| `mosaic` | uniform | 0.0 … 1.0 |
| `mixup` | uniform | 0.0 … 0.3 |

## Prerrequisitos (en tu Mac, antes de abrir este notebook)

1. **MLflow corriendo con SQLite backend:**
   ```bash
   mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 8080
   ```
2. **ngrok exponiendo el puerto 8080:**
   ```bash
   ngrok http --host-header=rewrite 8080
   ```
3. **Colab Secrets** configurados (ícono 🔑 en el panel izquierdo):
   - `GITHUB_REPO_URL` — URL HTTPS del repo
   - `MLFLOW_NGROK_URL` — URL de ngrok, ej. `https://aea2-xxxx.ngrok-free.app`
   - `WANDB_API_KEY` — Tu API key de W&B (opcional pero recomendado)
   - `ROBOFLOW_API_KEY` — Tu API key de Roboflow

## Orden de ejecución
Corre las celdas **en orden**. La búsqueda completa tarda ~4–5 horas en T4 con la config por defecto.

## 0. Verificar GPU

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(
    result.stdout
    if result.returncode == 0
    else "⚠️  No GPU detectada — actívala en Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU"
)

## 1. Leer secrets de Colab

In [ ]:
from google.colab import userdata

GITHUB_REPO_URL = userdata.get("GITHUB_REPO_URL")
MLFLOW_NGROK_URL = userdata.get("MLFLOW_NGROK_URL")
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")  # puede ser None si no tienes W&B

required = {
    "GITHUB_REPO_URL": GITHUB_REPO_URL,
    "MLFLOW_NGROK_URL": MLFLOW_NGROK_URL,
    "ROBOFLOW_API_KEY": ROBOFLOW_API_KEY,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise OSError(
        f"Faltan secrets en Colab: {missing}. Agrégalos en el panel 🔑 y vuelve a correr."
    )

print("✓ Secrets cargados")
print(f"  Repo:   {GITHUB_REPO_URL}")
print(f"  MLflow: {MLFLOW_NGROK_URL}")
print(f"  W&B:    {'configurado' if WANDB_API_KEY else 'no configurado (se desactivará)'}")

## 2. Instalar dependencias

In [ ]:
!pip install -q \
    'ultralytics>=8.3' \
    'mlflow>=3.12.0' \
    'optuna>=4.0' \
    'hydra-core>=1.3' \
    'omegaconf>=2.3' \
    'wandb>=0.18' \
    'loguru>=0.7' \
    'python-dotenv>=1.0' \
    'pyyaml>=6.0' \
    'roboflow>=1.1'
print('✓ Dependencias instaladas')

## 3. Clonar el repositorio

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("/content/football-player-tracker")

if REPO_DIR.exists():
    print("Repo ya existe, haciendo pull...")
    !git -C {REPO_DIR} pull
else:
    !git clone {GITHUB_REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"✓ Directorio de trabajo: {os.getcwd()}")

## 4. Descargar dataset desde Roboflow

In [ ]:
DATASET_DIR = REPO_DIR / "data" / "processed" / "football-players"

if DATASET_DIR.exists():
    n_train = len(list((DATASET_DIR / "train" / "images").glob("*")))
    print(f"✓ Dataset ya presente ({n_train} imágenes en train). Saltando descarga.")
else:
    print("Descargando dataset desde Roboflow...")
    !python scripts/download_dataset.py --api-key {ROBOFLOW_API_KEY}
    print("✓ Dataset descargado")

## 5. Configurar variables de entorno

In [ ]:
import os

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_NGROK_URL
os.environ["MLFLOW_EXPERIMENT_NAME"] = "football-player-tracker"
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

# W&B: configurar solo si tenemos la key, de lo contrario desactivar
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    wandb_enabled = "true"
else:
    os.environ["WANDB_MODE"] = "disabled"
    wandb_enabled = "false"

# Escribir .env para que python-dotenv lo pickee en el módulo de training
env_lines = [
    f"MLFLOW_TRACKING_URI={MLFLOW_NGROK_URL}",
    "MLFLOW_EXPERIMENT_NAME=football-player-tracker",
    f"ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}",
]
if WANDB_API_KEY:
    env_lines.append(f"WANDB_API_KEY={WANDB_API_KEY}")
(REPO_DIR / ".env").write_text("\n".join(env_lines) + "\n")

print("✓ Variables de entorno configuradas")
print(f"  MLFLOW_TRACKING_URI = {MLFLOW_NGROK_URL}")
print(f"  W&B habilitado       = {wandb_enabled}")

## 6. Verificar conectividad con MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_NGROK_URL)

try:
    experiments = mlflow.search_experiments()
    print(f"✓ Conexión OK — {len(experiments)} experimento(s) en el servidor")
    for exp in experiments:
        print(f"  • {exp.name} (id={exp.experiment_id})")
except Exception as e:
    print(f"✗ No se pudo conectar al MLflow en {MLFLOW_NGROK_URL}")
    print(f"  Error: {e}")
    print()
    print("Checklist:")
    print("  1. ¿Está corriendo mlflow ui --backend-store-uri sqlite:///mlflow.db en tu Mac?")
    print("  2. ¿Está corriendo ngrok http --host-header=rewrite 8080?")
    print("  3. ¿Copiaste bien la URL ngrok en el secret MLFLOW_NGROK_URL?")

## 7. Registrar el paquete como importable

In [ ]:
!pip install -e . --no-deps -q
print("✓ football_tracker registrado como importable")

## 8. Lanzar búsqueda Optuna

**20 trials × 15 épocas = ~300 épocas totales.**  
Tiempo estimado en T4: ~4–5 horas.

Mientras corre puedes ver el progreso en tu **MLflow UI** (`http://localhost:8080`):  
- Un run padre `hpo_football-yolov11-hpo` con el resumen del estudio.  
- Dentro, 20 runs hijos `trial_000`, `trial_001`, … con las métricas de cada trial.

> **Tip:** Para una prueba rápida primero, cambia `tuning.n_trials=3 tuning.trial_epochs=5`.

In [ ]:
# W&B desactivado en los trials (demasiado ruido para 20 runs cortos).
# El tracking completo va a MLflow como nested runs.
!python -m football_tracker.training.tune \
    tracking_backend.wandb.enabled=false \
    training.device=0 \
    training.workers=2 \
    training.batch=16

## 9. Resultados del estudio

Muestra el mejor trial y los top-5 según la métrica objetivo.

In [ ]:
import glob

import optuna

# Encontrar el archivo .db del estudio más reciente
db_files = sorted(glob.glob(str(REPO_DIR / "runs" / "**" / "optuna_study.db"), recursive=True))

if not db_files:
    print("⚠️  No se encontró optuna_study.db. ¿Terminó la búsqueda?")
else:
    latest_db = db_files[-1]
    print(f"Cargando estudio desde: {latest_db}")

    study = optuna.load_study(
        study_name="football-yolov11-hpo",
        storage=f"sqlite:///{latest_db}",
    )

    best = study.best_trial
    print(f"\n{'='*60}")
    print(f"Mejor trial: #{best.number}")
    print(f"Mejor valor: {best.value:.4f} (mAP50-95)")
    print("\nMejores hiperparámetros:")
    for k, v in best.params.items():
        print(f"  {k:20s} = {v}")
    print(f"{'='*60}")

    # Top-5 trials
    print("\nTop-5 trials:")
    trials_sorted = sorted(
        [t for t in study.trials if t.value is not None],
        key=lambda t: t.value,
        reverse=True,
    )
    for _i, t in enumerate(trials_sorted[:5]):
        print(
            f"  #{t.number:3d}  mAP50-95={t.value:.4f}  lr0={t.params.get('lr0', 'N/A'):.2e}  momentum={t.params.get('momentum', 'N/A'):.3f}"
        )

## 10. Reentrenar con los mejores hiperparámetros

Una vez identificados los mejores params, lanzamos un entrenamiento completo (30 épocas) con ellos.

In [ ]:
import optuna, glob

db_files = sorted(glob.glob(str(REPO_DIR / "runs" / "**" / "optuna_study.db"), recursive=True))
if not db_files:
    print("⚠️  Corre primero la celda de búsqueda HPO.")
else:
    latest_db = db_files[-1]
    study = optuna.load_study(
        study_name="football-yolov11-hpo",
        storage=f"sqlite:///{latest_db}",
    )
    best = study.best_trial
    bp = best.params

    # Construir overrides de Hydra con los mejores parámetros
    overrides = " ".join(f"training.{k}={v}" for k, v in bp.items())
    print(f"Lanzando entrenamiento completo con overrides:
  {overrides}
")

    !python -m football_tracker.training.train \
        training.epochs=30 \
        training.device=0 \
        training.workers=2 \
        training.batch=16 \
        {overrides}

## 11. Descargar best.pt optimizado

Descarga el mejor modelo del entrenamiento con los hiperparámetros optimizados.

In [ ]:
import glob
import shutil
from pathlib import Path

from google.colab import files

# Último best.pt creado (del retrain con mejores params)
candidates = sorted(
    glob.glob(str(REPO_DIR / "runs" / "**" / "weights" / "best.pt"), recursive=True)
)

if not candidates:
    print("⚠️  No se encontró best.pt.")
else:
    best_pt = Path(candidates[-1])
    dest_dir = REPO_DIR / "models" / "optimized"
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(best_pt, dest_dir / "best.pt")
    print(f"✓ best.pt copiado a {dest_dir / 'best.pt'} ({best_pt.stat().st_size / 1e6:.1f} MB)")
    files.download(str(dest_dir / "best.pt"))
    print("✓ Descarga iniciada. Guárdalo en models/optimized/ dentro de tu repo local.")

## 12. Siguientes pasos

1. Mueve `best.pt` a `models/optimized/best.pt` en tu repo local.
2. En tu MLflow UI (`http://localhost:8080`) verifica que el parent run `hpo_football-yolov11-hpo` tiene los 20 nested runs con las métricas de cada trial.
3. Abre la comparativa de runs en MLflow para ver el efecto de cada hiperparámetro.
4. Commit con los resultados:
   ```bash
   git add models/optimized/best.pt
   git commit -m 'feat(hpo): add optimized weights from Optuna search'
   git push
   ```
5. Con esto cerramos **Session 3 (parte 1 — HPO)**. La parte 2 es FiftyOne error analysis.